# 08_nlp_evaluation: BLEU and ROUGE Overlap Metrics Verification

This notebook validates NLP evaluation metrics. It implements BLEU precision scoring using NLTK and ROUGE recall scoring from scratch, verifying the math against our step-by-step hand calculations.

## 1. BLEU Score Precision and Brevity Penalty

In [1]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

# Define candidate and reference tokens matching hand-calculation
candidate = ["the", "cat", "sat"]
reference = [["the", "cat", "sat", "on", "the", "mat"]]

# Calculate BLEU-2 with weights (0.5, 0.5) and no smoothing
weights = (0.5, 0.5)
bleu_score = sentence_bleu(reference, candidate, weights=weights, smoothing_function=SmoothingFunction().method0)

print(f"NLTK BLEU-2 Score: {bleu_score:.4f}")

# Verify exact match with hand calculation score of 0.3679
np.testing.assert_almost_equal(bleu_score, 0.3679, decimal=4)

NLTK BLEU-2 Score: 0.3679


### Output Analysis: BLEU Metric
The computed BLEU-2 score is exactly `0.3679`. Because the candidate length $c=3$ is shorter than reference length $r=6$, the brevity penalty $\text{BP} = e^{1-6/3} = e^{-1} \approx 0.3679$ dampens the perfect unigram and bigram precisions ($p_1=1.0, p_2=1.0$), ensuring that models cannot cheat the evaluation metric by outputting short snippets.

## 2. ROUGE-1 Recall and F1 Score from Scratch

In [2]:
from collections import Counter

# Count unigram overlaps
cand_counts = Counter(candidate)
ref_counts = Counter(reference[0])

overlaps = 0
for word, count in cand_counts.items():
    overlaps += min(count, ref_counts[word])

recall_rouge = overlaps / len(reference[0])
precision_rouge = overlaps / len(candidate)
f1_rouge = 2 * (precision_rouge * recall_rouge) / (precision_rouge + recall_rouge)

print(f"ROUGE-1 Recall:    {recall_rouge:.4f}")
print(f"ROUGE-1 Precision: {precision_rouge:.4f}")
print(f"ROUGE-1 F1 Score:  {f1_rouge:.4f}")

# Verify consistency with hand calculations (Recall = 0.5, F1 = 0.6667)
np.testing.assert_almost_equal(recall_rouge, 0.5000, decimal=4)
np.testing.assert_almost_equal(f1_rouge, 0.6667, decimal=4)

ROUGE-1 Recall:    0.5000
ROUGE-1 Precision: 1.0000
ROUGE-1 F1 Score:  0.6667


### Output Analysis: ROUGE Metric
The ROUGE recall outputs match our hand calculations perfectly (Recall = `0.5000`, F1 = `0.6667`). This confirms that ROUGE focuses on coverage (measuring how much of the reference was captured), complementary to BLEU's precision focus.